# AI Answer Source-Role Drift Lab

A reproducible AEO/GEO notebook for detecting how the *role* of exposed sources changes across controlled answer-engine reruns. It keeps source-role movement, citation gains and losses, and target-domain citations separate instead of compressing them into one score.

**Synthetic demonstration:** every observation below is invented for method testing. It is not a claim about a real engine, customer, ranking, or Corank result. Replace it with retained answer-level evidence from your own audit.

Method companion: [Corank AI-search visibility workflow](https://corank.ai/).

## What this lab measures

For each prompt and observation window, retain the exact answer and record the most prominent exposed citation as one of four descriptive roles:

- `none`: no citation was exposed;
- `primary`: a first-party source for the supported claim;
- `independent`: a topically relevant third party that can corroborate the claim;
- `aggregator`: a roundup, directory, or summary source.

The roles are not a universal quality ranking. A shift is classified by what changed, and the recommended action depends on the transition. Keep engine, mode, locale, prompt text, timing, and capture method controlled before interpreting movement.

In [ ]:
from collections import Counter, defaultdict
import json

WINDOWS = ["baseline", "week_1", "week_2"]
ALLOWED_ROLES = {"none", "primary", "independent", "aggregator"}
STAGE_PRIORITY = {"decision": 1, "consideration": 2, "awareness": 3}

# Synthetic records only. Domains under .example are reserved for documentation.
OBSERVATIONS = [
    {"prompt_id": "p01", "stage": "decision", "intent": "comparison", "window": "baseline", "brand_mentioned": True,  "any_citation": True,  "target_cited": True,  "source_role": "primary",     "source_domain": "target.example"},
    {"prompt_id": "p01", "stage": "decision", "intent": "comparison", "window": "week_1",  "brand_mentioned": True,  "any_citation": True,  "target_cited": False, "source_role": "independent", "source_domain": "publisher.example"},
    {"prompt_id": "p01", "stage": "decision", "intent": "comparison", "window": "week_2",  "brand_mentioned": True,  "any_citation": True,  "target_cited": False, "source_role": "aggregator",  "source_domain": "roundup.example"},

    {"prompt_id": "p02", "stage": "decision", "intent": "evaluation", "window": "baseline", "brand_mentioned": False, "any_citation": False, "target_cited": False, "source_role": "none",        "source_domain": None},
    {"prompt_id": "p02", "stage": "decision", "intent": "evaluation", "window": "week_1",  "brand_mentioned": True,  "any_citation": True,  "target_cited": True,  "source_role": "primary",     "source_domain": "target.example"},
    {"prompt_id": "p02", "stage": "decision", "intent": "evaluation", "window": "week_2",  "brand_mentioned": True,  "any_citation": True,  "target_cited": True,  "source_role": "primary",     "source_domain": "target.example"},

    {"prompt_id": "p03", "stage": "consideration", "intent": "diagnosis", "window": "baseline", "brand_mentioned": True, "any_citation": True, "target_cited": False, "source_role": "independent", "source_domain": "research.example"},
    {"prompt_id": "p03", "stage": "consideration", "intent": "diagnosis", "window": "week_1",  "brand_mentioned": True, "any_citation": True, "target_cited": False, "source_role": "independent", "source_domain": "research.example"},
    {"prompt_id": "p03", "stage": "consideration", "intent": "diagnosis", "window": "week_2",  "brand_mentioned": True, "any_citation": True, "target_cited": True,  "source_role": "primary",     "source_domain": "target.example"},

    {"prompt_id": "p04", "stage": "awareness", "intent": "definition", "window": "baseline", "brand_mentioned": False, "any_citation": True, "target_cited": False, "source_role": "aggregator", "source_domain": "glossary.example"},
    {"prompt_id": "p04", "stage": "awareness", "intent": "definition", "window": "week_1",  "brand_mentioned": False, "any_citation": True, "target_cited": False, "source_role": "aggregator", "source_domain": "glossary.example"},
    {"prompt_id": "p04", "stage": "awareness", "intent": "definition", "window": "week_2",  "brand_mentioned": False, "any_citation": True, "target_cited": False, "source_role": "aggregator", "source_domain": "glossary.example"},

    {"prompt_id": "p05", "stage": "consideration", "intent": "implementation", "window": "baseline", "brand_mentioned": True, "any_citation": True,  "target_cited": True,  "source_role": "primary",     "source_domain": "target.example"},
    {"prompt_id": "p05", "stage": "consideration", "intent": "implementation", "window": "week_1",  "brand_mentioned": True, "any_citation": False, "target_cited": False, "source_role": "none",        "source_domain": None},
    {"prompt_id": "p05", "stage": "consideration", "intent": "implementation", "window": "week_2",  "brand_mentioned": True, "any_citation": True,  "target_cited": False, "source_role": "independent", "source_domain": "tutorial.example"},

    {"prompt_id": "p06", "stage": "awareness", "intent": "category", "window": "baseline", "brand_mentioned": False, "any_citation": False, "target_cited": False, "source_role": "none",        "source_domain": None},
    {"prompt_id": "p06", "stage": "awareness", "intent": "category", "window": "week_1",  "brand_mentioned": False, "any_citation": False, "target_cited": False, "source_role": "none",        "source_domain": None},
    {"prompt_id": "p06", "stage": "awareness", "intent": "category", "window": "week_2",  "brand_mentioned": True,  "any_citation": True,  "target_cited": False, "source_role": "independent", "source_domain": "association.example"},
]


In [ ]:
def validate_observations(rows):
    issues = []
    keys = [(row["prompt_id"], row["window"]) for row in rows]
    if len(keys) != len(set(keys)):
        issues.append("Duplicate prompt-window keys found.")

    by_prompt = defaultdict(list)
    for row in rows:
        by_prompt[row["prompt_id"]].append(row)
        if row["source_role"] not in ALLOWED_ROLES:
            issues.append(f"{row['prompt_id']} {row['window']}: invalid source role")
        if row["target_cited"] and not row["any_citation"]:
            issues.append(f"{row['prompt_id']} {row['window']}: target citation without any citation")
        if row["any_citation"] != (row["source_role"] != "none"):
            issues.append(f"{row['prompt_id']} {row['window']}: citation flag and role disagree")
        if row["any_citation"] != bool(row["source_domain"]):
            issues.append(f"{row['prompt_id']} {row['window']}: citation flag and domain disagree")
        if row["target_cited"] != (row["source_domain"] == "target.example"):
            issues.append(f"{row['prompt_id']} {row['window']}: target flag and synthetic domain disagree")

    for prompt_id, prompt_rows in by_prompt.items():
        observed = {row["window"] for row in prompt_rows}
        if observed != set(WINDOWS):
            issues.append(f"{prompt_id}: expected {WINDOWS}, found {sorted(observed)}")
    return issues

validation_issues = validate_observations(OBSERVATIONS)
assert not validation_issues, validation_issues
print(f"Validated {len(OBSERVATIONS)} synthetic observations across {len(set(r['prompt_id'] for r in OBSERVATIONS))} prompts.")


In [ ]:
def classify_transition(previous, current):
    old_role, new_role = previous["source_role"], current["source_role"]
    if old_role == new_role:
        return "stable"
    if new_role == "none":
        return "citation_loss"
    if old_role == "none":
        return "new_citation"
    if new_role == "aggregator":
        return "to_aggregator"
    if new_role == "primary":
        return "to_primary"
    if new_role == "independent":
        return "to_independent"
    raise ValueError(f"Unhandled role transition: {old_role} -> {new_role}")

by_prompt = defaultdict(list)
for observation in OBSERVATIONS:
    by_prompt[observation["prompt_id"]].append(observation)

transitions = []
for prompt_id, rows in sorted(by_prompt.items()):
    ordered = sorted(rows, key=lambda row: WINDOWS.index(row["window"]))
    for previous, current in zip(ordered, ordered[1:]):
        transitions.append({
            "prompt_id": prompt_id,
            "stage": current["stage"],
            "intent": current["intent"],
            "from_window": previous["window"],
            "to_window": current["window"],
            "from_role": previous["source_role"],
            "to_role": current["source_role"],
            "transition": classify_transition(previous, current),
            "target_gain": (not previous["target_cited"] and current["target_cited"]),
            "target_loss": (previous["target_cited"] and not current["target_cited"]),
        })

print("prompt | window change        | role change                 | classification")
print("-" * 86)
for row in transitions:
    window_change = f"{row['from_window']} -> {row['to_window']}"
    role_change = f"{row['from_role']} -> {row['to_role']}"
    print(f"{row['prompt_id']:6} | {window_change:20} | {role_change:27} | {row['transition']}")


In [ ]:
ACTIONS = {
    "citation_loss": "Preserve the lost citation evidence, test anonymous retrieval of the intended source, then rerun unchanged conditions.",
    "new_citation": "Archive the new citation and rerun before treating it as repeatable.",
    "to_aggregator": "Compare claim coverage with the aggregator, strengthen claim-level primary evidence, and seek legitimate independent corroboration.",
    "to_primary": "Verify attribution, freshness, and claim accuracy; retain the evidence and monitor repeatability.",
    "to_independent": "Verify that the independent source actually supports the claim and document its relationship to the entity.",
    "stable": "Keep monitoring; stable role does not prove stable wording, prominence, accuracy, or business impact.",
}

transition_counts = Counter(row["transition"] for row in transitions)
target_gains = sum(row["target_gain"] for row in transitions)
target_losses = sum(row["target_loss"] for row in transitions)

queue = sorted(
    transitions,
    key=lambda row: (
        STAGE_PRIORITY[row["stage"]],
        0 if row["transition"] in {"citation_loss", "to_aggregator"} else 1,
        row["prompt_id"],
        WINDOWS.index(row["to_window"]),
    ),
)

print("Transition counts:", dict(sorted(transition_counts.items())))
print("Target-domain citation gains:", target_gains)
print("Target-domain citation losses:", target_losses)
print("\nStage-ordered action queue:")
for row in queue:
    priority = f"P{STAGE_PRIORITY[row['stage']]}"
    print(f"- {priority} {row['prompt_id']} {row['from_window']}->{row['to_window']} [{row['transition']}]: {ACTIONS[row['transition']]}")


## Interpretation guardrails

1. A role transition is an observation, not proof that a specific SEO change caused it.
2. A primary source is not automatically accurate, and an independent source is not automatically unbiased. Audit the cited claim.
3. Preserve losses and unfavorable reruns; do not publish only the best answer.
4. Segment decision, consideration, and awareness prompts before prioritizing work.
5. Connect citation observations to qualified demand separately. This notebook does not measure revenue or guarantee rankings.

In [ ]:
report = {
    "data_status": "synthetic demonstration",
    "observation_count": len(OBSERVATIONS),
    "prompt_count": len(by_prompt),
    "transition_count": len(transitions),
    "transition_counts": dict(sorted(transition_counts.items())),
    "target_citation_gains": target_gains,
    "target_citation_losses": target_losses,
    "validation_issues": validation_issues,
}
print(json.dumps(report, indent=2, sort_keys=True))


## Extend the method

Add the retained answer URL or artifact ID, engine and mode, locale, capture timestamp, exact cited URL, claim supported, and reviewer decision. Keep personally identifiable or confidential prompt data out of public notebooks.

For a broader evidence-led AEO/GEO workflow, visit [Corank](https://corank.ai/).